# Feature Engineering

Buckets rare estates, builds the stratified train/test split, and previews the model-ready features used for training.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

In [2]:
from src.data.load_data import load_config, load_raw_data
from src.data.preprocess import clean_data
from src.features.feature_engineering import bucket_rare_estates, split_data, split_X_y

config = load_config()
clean = clean_data(load_raw_data(config))

## Estate frequency before bucketing

37 distinct estates exist in the cleaned data, but the distribution is heavily skewed - a handful of estates dominate, and many appear only once or twice. One-hot encoding all 37 directly would create several columns with almost no signal, and risks an estate appearing in only one of the train/test splits.

In [3]:
clean["Estate"].value_counts()

Estate
Dagoretti North        715
Westlands              547
Kiambaa                 60
Nyali                   41
Kiambu Road             37
Langata                 36
Ngong Road              17
Roysambu                11
Athi River              10
Thindigua                9
Dagoretti South          7
Embakasi                 7
Kajiado North            6
Kikuyu                   5
Thika Road               5
Mombasa Road             4
Milimani                 4
Kisauni                  3
Ruiru                    3
Kabete                   3
Starehe                  3
Mombasa CBD              3
Muthaiga                 3
Kasarani                 2
Ngong                    2
Kisumu Central           2
Kiambu Constituency      2
Thika                    1
Kisumu West              1
Thika East               1
Kilifi South             1
Kiambu Town              1
Eldoret North            1
Makadara                 1
Kangundo                 1
Mvita                    1
Ruaraka              

## Bucketed estates

Estates with fewer than `rare_estate_threshold` listings (currently 10, set in `config.yaml`) are grouped into a single `Other` category before encoding.

In [4]:
min_count = config["features"]["rare_estate_threshold"]
bucketed = bucket_rare_estates(clean, min_count)
bucketed["Estate"].value_counts()

Estate
Dagoretti North    715
Westlands          547
Other               83
Kiambaa             60
Nyali               41
Kiambu Road         37
Langata             36
Ngong Road          17
Roysambu            11
Athi River          10
Name: count, dtype: int64

Bucketing brings 37 estates down to 10 categories: 9 named estates that individually clear the threshold, plus `Other` absorbing the remaining 28 rare estates (83 rows, ~5% of the data). `Dagoretti North` and `Westlands` alone still account for the large majority of listings - bucketing addresses the long tail of rare estates without distorting the two dominant categories that were already well represented.

## Stratified train/test split

The split is stratified on the bucketed `Estate` column, so that even the smallest retained categories are proportionally represented in both the training and test sets, rather than risking a rare category landing almost entirely in one split by chance.

In [5]:
train_df, test_df = split_data(clean, config)
X_train, y_train = split_X_y(train_df)
X_test, y_test = split_X_y(test_df)

print("Train:", X_train.shape, "Test:", X_test.shape)
X_train.head()

Train: (1245, 11) Test: (312, 11)


,Bedrooms,Bathrooms,Estate_Dagoretti North,Estate_Kiambaa,Estate_Kiambu Road,Estate_Langata,Estate_Ngong Road,Estate_Nyali,Estate_Other,Estate_Roysambu,Estate_Westlands
1518,4,3,False,False,False,False,False,False,False,False,True
1198,4,4,False,False,False,False,False,False,False,False,True
98,2,2,False,False,False,False,False,False,True,False,False
564,3,3,False,False,False,False,False,False,False,False,True
1195,3,3,True,False,False,False,False,False,False,False,False


1,245 training rows and 312 test rows (80/20 of 1,557), 11 feature columns: `Bedrooms`, `Bathrooms`, and 9 one-hot `Estate_*` columns. Only 9 dummy columns appear even though 10 categories exist - `Estate_Athi River` is intentionally dropped as the baseline category (`pd.get_dummies` with `drop_first=True` drops the alphabetically-first category) to avoid perfect multicollinearity in Linear Regression; a row with all 9 `Estate_*` columns as `False` is implicitly an Athi River listing.